In [ ]:
# Collect every figure the recording speaks, into one JSON file, so the script and the
# tables cannot disagree. Written to Files/ so it can be pulled down without a portal.
import json

import notebookutils
from pyspark.sql import functions as F

_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_ID = {}


def lake_table(lakehouse, table):
    if lakehouse not in _ID:
        _ID[lakehouse] = notebookutils.lakehouse.get(lakehouse, workspaceId=_WS).id
    return spark.read.format("delta").load(
        f"abfss://{_WS}@{_ONELAKE}/{_ID[lakehouse]}/Tables/{table}")


out = {}
state = spark.table("gold_referral_state")
out["cohort"] = state.count()
for row in state.groupBy("referral_state").count().collect():
    out[f"state_{row['referral_state']}"] = row["count"]
out["surfaced_by_notes_only"] = state.filter("surfaced_by_notes_only").count()

for row in spark.table("gold_notes_uplift").collect():
    g = row["group"]
    out[f"uplift_{g}_coded"] = row["surfaced_coded_only"]
    out[f"uplift_{g}_notes"] = row["surfaced_with_notes"]
    out[f"uplift_{g}_added"] = row["added_by_notes"]

for row in spark.table("gold_validation_evidence_uplift").collect():
    out[f"sens_{row['evidence_set']}_{row['group']}"] = round(row["sensitivity"] * 100, 1)

for row in lake_table("silver_lakehouse", "silver_extraction_quality").collect():
    out[f"extract_{row['metric']}"] = row["value"]

notes = lake_table("bronze_lakehouse", "bronze_clinical_notes")
out["notes_total"] = notes.count()
ext = lake_table("silver_lakehouse", "silver_extracted_findings")
out["extracted_total"] = ext.count()
for row in ext.groupBy("assertion").count().collect():
    out[f"extracted_{row['assertion']}"] = row["count"]

obs = lake_table("silver_lakehouse", "silver_observations")
out["coded_observations"] = obs.count()
# gold_observations must be the union -- the graph and the latency replay both
# read it, and if it is coded-only they silently disagree with the criteria.
gold_obs = spark.table("gold_observations")
out["gold_observations"] = gold_obs.count()
for row in gold_obs.groupBy("evidence_source").count().collect():
    out[f"gold_observations_{row['evidence_source']}"] = row["count"]

try:
    lat = spark.table("gold_signal_latency")
    out["latency_median_months"] = round(
        lat.approxQuantile("latency_months", [0.5], 0.001)[0], 1)
    out["latency_over_12m_pct"] = round(
        100.0 * lat.filter("latency_months >= 12").count() / lat.count(), 0)
    out["latency_max_months"] = round(
        lat.agg(F.max("latency_months")).collect()[0][0], 1)
    out["latency_rows"] = lat.count()
    for row in (lat.join(spark.table("gold_referral_state")
                         .select("patient_id", "interpreter_required"), "patient_id")
                .groupBy("interpreter_required").agg(
                    F.expr("percentile_approx(latency_months, 0.5)").alias("m"))
                .collect()):
        key = "interpreter" if row["interpreter_required"] else "no_interpreter"
        out[f"latency_median_{key}"] = round(row["m"], 1)
except Exception as exc:
    out["latency_error"] = str(exc)[:200]

# One patient who needed the notes -- the recording shows this row.
example = (state.filter("surfaced_by_notes_only")
           .select("patient_id", "criteria").limit(1).collect())
if example:
    out["example_notes_only_patient"] = example[0]["patient_id"]
    out["example_notes_only_criteria"] = list(example[0]["criteria"])

notebookutils.fs.put("Files/report/numbers.json", json.dumps(out, indent=2), True)
for k, v in sorted(out.items()):
    print(f"{k:38} {v}")
print("\nwrote Files/report/numbers.json")